[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab04_medical_image_classification.ipynb)

# Lab 4: การเรียนรู้เชิงลึก (Deep Learning) สำหรับการจำแนกภาพทางการแพทย์

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันพฤหัสบดีที่ 8 ตุลาคม 2026 · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สร้างและฝึกโครงข่ายประสาทเทียมแบบคอนโวลูชัน (convolutional neural network: CNN) ด้วยการเรียนรู้แบบถ่ายโอน (transfer learning) กับภาพถ่ายรังสีทรวงอก
- ใช้ชุดข้อมูล (dataset) สาธารณะ MedMNIST ซึ่งไม่ต้องดาวน์โหลดด้วยตนเองและใช้งานบน Colab ได้ทันที
- ประเมินผลด้วยเมทริกซ์ความสับสน (confusion matrix) ความไว (sensitivity) และความจำเพาะ (specificity) รวมทั้งพิจารณาภาพที่โมเดลจำแนกผิด
- เข้าใจภาพรวมของเครื่องมือกำกับข้อมูลภาพ (image labeling) เช่น LabelMe และ CVAT และการแบ่งส่วนภาพ (segmentation)

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ ด้านซ้ายของเซลล์ หรือกด **Shift+Enter**) แบบฝึกหัด ⭐ **Exercise** แต่ละข้อมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ผู้เรียนควรทดลองทำด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งถือเป็นชิ้นงานที่ต้องส่ง
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่นับเป็นส่วนของชิ้นงานที่ต้องส่ง
> - หากพบปัญหาหรือข้อผิดพลาด (error) สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

> **⚡ การเปิดใช้ GPU (โปรดดำเนินการก่อนรันเซลล์ใด ๆ)**
> การฝึกโครงข่ายประสาทเทียมบน GPU (หน่วยประมวลผลกราฟิก ซึ่งคำนวณแบบขนานได้จำนวนมาก) ใช้เวลาน้อยกว่าบน CPU อย่างมาก
> ใน Colab ให้เลือก **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
> การเปลี่ยน runtime จะเริ่มโน้ตบุ๊กใหม่ จึงควรดำเนินการ *ก่อน* รันเซลล์แรก (หรือรันใหม่ตั้งแต่ต้นหลังเปลี่ยน)
> หาก Colab แจ้งว่าไม่มี GPU ให้ใช้งาน ปฏิบัติการนี้ยังสามารถรันบน CPU ได้ แต่การฝึกโมเดลจะใช้เวลานานขึ้นหลายนาที

ปฏิบัติการทั้งหมดรันได้บน [Google Colab](https://colab.research.google.com) รุ่นที่ไม่มีค่าใช้จ่าย ตารางอันดับ (leaderboard) 🏆 ของรายวิชาอยู่ที่ [Hugging Face](https://huggingface.co/spaces/wdittaya/aidh-leaderboard) ผู้เรียนจำเป็นต้องระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่าเฉพาะเมื่อต้องการเข้าร่วมโจทย์ท้าทายเท่านั้น

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab04"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ส่วน A — ชุดข้อมูล PneumoniaMNIST (MedMNIST)

ภาพถ่ายรังสีทรวงอกของผู้ป่วยเด็กซึ่งกำกับว่า **ปกติ (normal, 0)** หรือ **ปอดอักเสบ (pneumonia, 1)** โดยย่อขนาดเหลือ 28×28 พิกเซล ชุดข้อมูลนี้มีขนาดเล็ก อยู่ในรูปแบบมาตรฐาน และติดตั้งได้ด้วย `pip` จึงเหมาะกับ GPU ของ Colab รุ่นที่ไม่มีค่าใช้จ่าย

เซลล์ถัดไปติดตั้งไลบรารีและแสดงอุปกรณ์ที่ใช้ประมวลผล ได้แก่ GPU (`cuda`) หรือ CPU

In [ ]:
!pip -q install medmnist
import torch, torch.nn as nn
import numpy as np, matplotlib.pyplot as plt
from medmnist import PneumoniaMNIST, INFO

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("อุปกรณ์:", device, "(GPU)" if device == "cuda" else "(CPU - ใช้งานได้แต่ช้ากว่า โปรดดูวิธีเปิดใช้ GPU ด้านบน)")

In [ ]:
# ดาวน์โหลดข้อมูล 3 ชุด: train (สำหรับฝึกโมเดล), val (สำหรับตรวจสอบระหว่างฝึก), test (สำหรับประเมินผลขั้นสุดท้าย)
from torchvision import transforms
tf = transforms.Compose([
    transforms.ToTensor(),              # ค่าพิกเซล 0-255 -> ตัวเลข 0-1
    transforms.Normalize([.5], [.5]),   # -> ตัวเลข -1 ถึง 1 (ช่วงค่าที่เหมาะกับโครงข่าย)
])
train_ds = PneumoniaMNIST(split="train", transform=tf, download=True, size=28)
val_ds   = PneumoniaMNIST(split="val",   transform=tf, download=True, size=28)
test_ds  = PneumoniaMNIST(split="test",  transform=tf, download=True, size=28)
print(INFO["pneumoniamnist"]["label"])
print("train/val/test:", len(train_ds), len(val_ds), len(test_ds))
for name, ds in [("train", train_ds), ("test", test_ds)]:
    n_pneu = int(ds.labels.sum())
    print(f"{name:5s}: ปกติ {len(ds)-n_pneu} ภาพ, ปอดอักเสบ {n_pneu} ภาพ (ปอดอักเสบ {n_pneu/len(ds):.0%})")

In [ ]:
# แสดงตัวอย่างภาพ (ข้อความบนกราฟเป็นภาษาอังกฤษ เนื่องจากฟอนต์เริ่มต้นของกราฟไม่รองรับภาษาไทย)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
for ax, (img, label) in zip(axes, train_ds):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title("pneumonia" if int(label[0]) == 1 else "normal", fontsize=9)
    ax.axis("off")
plt.show()

💡 **คำอธิบายผลลัพธ์** ชุดข้อมูลประกอบด้วยภาพถ่ายรังสีที่มีการกำกับข้อมูลแล้วประมาณ 5,900 ภาพ แบ่งเป็น *train* (ใช้ฝึกโมเดล) *validation* (ใช้ตรวจสอบระหว่างฝึก) และ *test* (624 ภาพ แยกไว้สำหรับประเมินผลขั้นสุดท้าย) ภาพในชุดฝึกประมาณสามในสี่เป็นภาพปอดอักเสบ กล่าวคือ ข้อมูลมี **ความไม่สมดุลของคลาส (class imbalance)** โมเดลที่ตอบว่า "ปอดอักเสบ" ทุกภาพจึงถูกต้องประมาณร้อยละ 75 ของชุดฝึก นอกจากนี้ ภาพขนาด 28×28 พิกเซลมีความละเอียดต่ำกว่าภาพถ่ายรังสีจริงมาก ชุดข้อมูลนี้จึงเหมาะสำหรับการเรียนการสอน ไม่ใช่การวินิจฉัย

In [ ]:
# ป้อนภาพให้โครงข่ายเป็นชุดย่อย (batch)
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128)
test_loader  = DataLoader(test_ds, batch_size=128)   # shuffle=False -> คงลำดับ 0..623

## การเรียนรู้แบบถ่ายโอน (transfer learning) ด้วย ResNet-18

**ResNet-18** เป็น CNN ที่ได้รับการฝึกมาแล้วด้วยภาพถ่ายทั่วไปประมาณ 1 ล้านภาพ (ชุดข้อมูล *ImageNet*) จึงสามารถตรวจจับขอบ พื้นผิว และรูปทรงได้ **การเรียนรู้แบบถ่ายโอน** คือการนำความรู้ดังกล่าวมาใช้ต่อ และปรับโครงข่ายเพียงบางส่วนให้เหมาะกับงานใหม่ วิธีนี้ทำให้ได้โมเดลที่มีประสิทธิภาพแม้มีข้อมูลจำนวนจำกัด จึงเป็นแนวทางมาตรฐานในงานภาพทางการแพทย์ ซึ่งภาพที่กำกับข้อมูลแล้วมีต้นทุนสูง

การปรับโครงข่ายมี 2 ส่วน ได้แก่
1. **ชั้นนำเข้า (input layer)** — ภาพ ImageNet เป็นภาพสี (3 ช่องสี (channel): แดง เขียว น้ำเงิน) ขณะที่ภาพถ่ายรังสีเป็นภาพระดับเทา (grayscale) ซึ่งมีเพียง **1** ช่อง
2. **ชั้นส่งออก (output layer)** — ImageNet มี 1,000 คลาส แต่งานนี้มีเพียง **2** คลาส (ปกติ และปอดอักเสบ)

**⭐ แบบฝึกหัดที่ 1 (Exercise 1):** ให้แทนที่ `...` ทั้งสองตำแหน่งในเซลล์ด้านล่างด้วยตัวเลขที่ถูกต้อง
*คำใบ้:* ภาพถ่ายรังสีระดับเทามีจำนวนช่องสีเท่าใด และโมเดลนี้มีคำตอบที่เป็นไปได้กี่แบบ

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ปรับโครงข่ายที่ฝึกมาแล้ว (ระบุตัวเลข 2 ค่า)
from torchvision import models
net = models.resnet18(weights="IMAGENET1K_V1")   # ดาวน์โหลดโครงข่ายที่ฝึกมาแล้ว

IN_CHANNELS = ...   # TODO: จำนวนช่องสีของภาพถ่ายรังสี (ภาพสี = 3, ภาพระดับเทา = ?)
N_CLASSES   = ...   # TODO: จำนวนคำตอบที่เป็นไปได้ (ปกติ / ปอดอักเสบ)

# ก) ชั้นแรกใหม่ที่รับภาพ IN_CHANNELS ช่อง โดยคงตัวกรองเดิมไว้ด้วยการรวมตัวกรอง RGB
old_w = net.conv1.weight.data.clone()
net.conv1 = nn.Conv2d(IN_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
if IN_CHANNELS == 1:
    net.conv1.weight.data = old_w.sum(dim=1, keepdim=True)
# ข) ชั้นสุดท้ายใหม่ที่มีเอาต์พุต N_CLASSES ค่า
net.fc = nn.Linear(net.fc.in_features, N_CLASSES)

net = net.to(device)
print(net.conv1, net.fc, sep="\n")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
from torchvision import models
net = models.resnet18(weights="IMAGENET1K_V1")

IN_CHANNELS = 1     # ภาพถ่ายรังสีระดับเทา = 1 ช่อง
N_CLASSES   = 2     # ปกติ / ปอดอักเสบ

old_w = net.conv1.weight.data.clone()
net.conv1 = nn.Conv2d(IN_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
if IN_CHANNELS == 1:
    net.conv1.weight.data = old_w.sum(dim=1, keepdim=True)
net.fc = nn.Linear(net.fc.in_features, N_CLASSES)

net = net.to(device)
print(net.conv1, net.fc, sep="\n")

💡 **คำอธิบายผลลัพธ์** โครงข่ายที่ฝึกมาแล้วได้รับการปรับ 2 ส่วน คือ ชั้นแรกรับภาพ 1 ช่อง (โดยเริ่มต้นจากตัวกรองภาพสีเดิม) และชั้นสุดท้ายมีเอาต์พุต 2 ค่า ส่วนชั้นกลางทั้งหมดยังคงความรู้ที่เรียนจาก ImageNet

## การฝึกโมเดล (โค้ดเตรียมไว้แล้ว ให้รันเซลล์)

หนึ่ง **รอบการฝึก (epoch)** หมายถึงการที่โมเดลประมวลผลภาพฝึกครบทุกภาพหนึ่งครั้ง และปรับพารามิเตอร์ภายในเพื่อลดความคลาดเคลื่อน ปฏิบัติการนี้ใช้เพียง **3 epoch** เพื่อจำกัดเวลา (ประมาณ 1 นาทีบน GPU หรือ 3–5 นาทีบน CPU)

In [ ]:
# ลูปการฝึก (โค้ดเตรียมไว้แล้ว)
import torch.optim as optim
EPOCHS = 3

def run_epoch(model, loader, opt=None):
    """ประมวลผลข้อมูลหนึ่งรอบ: หากระบุ optimizer จะฝึกโมเดล หากไม่ระบุจะประเมินผลเท่านั้น"""
    train = opt is not None
    crit = nn.CrossEntropyLoss()
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.view(-1).long().to(device)
        with torch.set_grad_enabled(train):
            out = model(x)
            loss = crit(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item() * len(x)
        correct += (out.argmax(1) == y).sum().item(); total += len(x)
    return loss_sum / total, correct / total

def train_model(model, train_loader, val_loader, epochs, lr=1e-3):
    opt = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        tl, ta = run_epoch(model, train_loader, opt)
        vl, va = run_epoch(model, val_loader)
        print(f"epoch {epoch+1}/{epochs}: train_acc={ta:.3f}  val_acc={va:.3f}")
    return model

net = train_model(net, train_loader, val_loader, EPOCHS)

💡 **คำอธิบายผลลัพธ์** *train_acc* คือความแม่นยำ (accuracy) บนภาพที่ใช้ฝึก ส่วน *val_acc* คือความแม่นยำบนภาพที่โมเดลไม่ได้ใช้ฝึก ซึ่งสะท้อนประสิทธิภาพจริงได้ดีกว่า หาก train_acc สูงกว่า val_acc อย่างมาก แสดงว่าโมเดลเกิด **การเรียนรู้เกิน (overfitting)** คือจดจำข้อมูลฝึกแทนการเรียนรู้รูปแบบทั่วไป ผลลัพธ์ของผู้เรียนแต่ละคนอาจแตกต่างกันเพียงเล็กน้อย เนื่องจากการฝึกมีองค์ประกอบเชิงสุ่ม

## การประเมินผลบนชุดทดสอบ (test set)

เซลล์ถัดไปให้โมเดลคำนวณ **ความน่าจะเป็นของภาวะปอดอักเสบ** สำหรับภาพทดสอบทั้ง 624 ภาพ โดยค่าเริ่มต้น ความน่าจะเป็นตั้งแต่ 0.5 ขึ้นไปหมายถึง "ทำนายว่าเป็นปอดอักเสบ"

In [ ]:
# ให้โมเดลทำนายชุดทดสอบ (TEST) (โค้ดเตรียมไว้แล้ว)
net.eval()
test_probs, y_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        test_probs += torch.softmax(net(x.to(device)), dim=1)[:, 1].cpu().tolist()
        y_true += y.view(-1).long().tolist()
test_probs, y_true = np.array(test_probs), np.array(y_true)
y_pred = (test_probs >= 0.5).astype(int)
print(f"ความแม่นยำบนชุดทดสอบ: {(y_pred == y_true).mean():.3f}   (หากตอบ 'ปอดอักเสบ' ทุกภาพ จะได้ {y_true.mean():.3f})")

💡 **ข้อสังเกต** ความแม่นยำบนชุดทดสอบมักต่ำกว่า val_acc อย่างชัดเจน เนื่องจากชุดทดสอบของ PneumoniaMNIST มาจากกลุ่มผู้ป่วยที่แตกต่างจากชุดฝึกและมีสัดส่วนภาพปกติมากกว่า สถานการณ์นี้เทียบได้กับการนำโมเดลไปใช้ในโรงพยาบาลอื่น จึงจำเป็นต้องประเมินโมเดลกับข้อมูลใหม่ที่แยกออกมาอย่างแท้จริงเสมอ

### การอ่านเมทริกซ์ความสับสน (confusion matrix)

**Confusion matrix** แสดงจำนวนผลลัพธ์ 4 ประเภท (ผลบวก (positive) = ปอดอักเสบ)

| | โมเดลทำนายว่า **ปกติ** | โมเดลทำนายว่า **ปอดอักเสบ** |
|---|---|---|
| **ปกติจริง** | TN — ผลลบจริง (true negative) ✔ | FP — ผลบวกลวง (false positive) |
| **ปอดอักเสบจริง** | FN — ผลลบลวง (false negative) คือ **ผู้ป่วยที่ตรวจไม่พบ** | TP — ผลบวกจริง (true positive) ✔ |

- **ความไว (sensitivity, recall)** = สัดส่วนของผู้ป่วยปอดอักเสบจริงที่โมเดลตรวจพบ = TP / (TP + FN)
- **ความจำเพาะ (specificity)** = สัดส่วนของผู้ที่ปกติจริงที่โมเดลทำนายว่าปกติได้ถูกต้อง = TN / (TN + FP)

**⭐ แบบฝึกหัดที่ 2 (Exercise 2):** ให้เขียนสูตร 2 บรรทัดท้ายเซลล์ โดยใช้ตัวแปร `tp`, `tn`, `fp`, `fn`
*คำใบ้:* สามารถคัดลอกสูตรจากรายการข้างต้น เช่น `sensitivity = tp / (tp + fn)`

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: ความไวและความจำเพาะจาก confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

sensitivity = ...   # TODO: TP / (TP + FN)
specificity = ...   # TODO: TN / (TN + FP)
print(f"ความไว (sensitivity)={sensitivity:.3f}  ความจำเพาะ (specificity)={specificity:.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"ความไว (sensitivity)={sensitivity:.3f}  ความจำเพาะ (specificity)={specificity:.3f}")

💡 **คำอธิบายผลลัพธ์** ความแม่นยำเพียงค่าเดียวไม่ได้แสดงว่าโมเดลผิดพลาด *ในลักษณะใด* ขณะที่ confusion matrix แสดงได้ชัดเจน ผลที่พบบ่อยกับชุดข้อมูลนี้คือความไวสูงมากแต่ความจำเพาะต่ำกว่า กล่าวคือ โมเดลมีแนวโน้มทำนายว่าเป็นปอดอักเสบมากเกินจริง ส่วนหนึ่งเนื่องจากภาพฝึกส่วนใหญ่เป็นภาพปอดอักเสบ

✍️ **คำถามที่ 1** จาก confusion matrix ข้างต้น (ก) โปรดระบุจำนวนผู้ป่วยเด็กปอดอักเสบที่โมเดล **ตรวจไม่พบ** (ข) โปรดระบุจำนวนเด็กปกติที่ได้รับ **ผลบวกลวง** (ค) จงอธิบายผลที่จะเกิดขึ้นกับผู้ป่วยในแต่ละกรณีในเวชปฏิบัติจริง และระบุว่าความผิดพลาดประเภทใดมีความรุนแรงมากกว่า เพราะเหตุใด

✍️ **คำตอบ:**



## การพิจารณาภาพที่โมเดลจำแนกผิด (โค้ดเตรียมไว้แล้ว ให้รันเซลล์)

ตัวเลขเชิงสถิติเพียงอย่างเดียวไม่เพียงพอ บุคลากรทางการแพทย์ควรพิจารณาด้วยว่าโมเดลผิดพลาดกับภาพ *ใด* เซลล์ด้านล่างแสดง **ผู้ป่วยปอดอักเสบที่โมเดลตรวจไม่พบ** (ผลลบลวง) และ **ผลบวกลวง** ที่โมเดลมีความมั่นใจสูงที่สุด พร้อมความน่าจะเป็นของภาวะปอดอักเสบที่โมเดลคำนวณ (`P(pneu)`) และแสดงภาพที่จำแนกถูกต้องเพื่อเปรียบเทียบ หากต้องการแสดงภาพเพิ่ม ให้เปลี่ยนค่า `N_SHOW`

In [ ]:
# แสดงภาพที่โมเดลจำแนกผิดโดยมีความมั่นใจสูงที่สุด (โค้ดเตรียมไว้แล้ว)
N_SHOW = 8

def show_cases(idx, title):
    idx = list(idx)[:N_SHOW]
    if not idx:
        print(f"{title}: ไม่พบกรณีดังกล่าว"); return
    fig, axes = plt.subplots(1, len(idx), figsize=(2 * len(idx), 2.6), squeeze=False)
    for ax, i in zip(axes[0], idx):
        ax.imshow(test_ds[i][0].squeeze(), cmap="gray")
        ax.set_title(f"#{i}  P(pneu)={test_probs[i]:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title); plt.show()

fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]
fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
show_cases(fn_idx[np.argsort(test_probs[fn_idx])],  f"Missed pneumonia (false negatives): {len(fn_idx)} in total")
show_cases(fp_idx[np.argsort(-test_probs[fp_idx])], f"False alarms (false positives): {len(fp_idx)} in total")
tp_idx = np.where((y_true == 1) & (y_pred == 1))[0]
tn_idx = np.where((y_true == 0) & (y_pred == 0))[0]
show_cases(tp_idx[np.argsort(-test_probs[tp_idx])], "For comparison - correctly detected pneumonia")
show_cases(tn_idx[np.argsort(test_probs[tn_idx])],  "For comparison - correctly cleared normals")

✍️ **คำถามที่ 2** จากการเปรียบเทียบภาพที่จำแนกผิดกับภาพที่จำแนกถูกต้อง ผู้เรียนพิจารณาว่าปัจจัยใดอาจทำให้โมเดลผิดพลาด (เช่น คุณภาพภาพ การจัดท่าผู้ป่วย ความผิดปกติที่ไม่ชัดเจน หรือความละเอียดที่ต่ำมากเพียง 28×28 พิกเซล) และผู้เรียนจะเชื่อถือความน่าจะเป็น 0.95 มากกว่า 0.55 หรือไม่ เพราะเหตุใด

✍️ **คำตอบ:**



## การเลือกเกณฑ์ตัดสิน (threshold)

โมเดลให้ผลลัพธ์เป็นความน่าจะเป็น ส่วนจุดตัดเป็นสิ่งที่ผู้ใช้งานกำหนด การลดเกณฑ์ตัดสิน (เช่น 0.3) ทำให้ภาพจำนวนมากขึ้นถูกจำแนกว่าเป็นปอดอักเสบ ส่งผลให้ **ความไวเพิ่มขึ้นและความจำเพาะลดลง** ส่วนการเพิ่มเกณฑ์ (เช่น 0.8) ให้ผลในทางตรงข้าม ค่าที่เหมาะสมขึ้นกับวัตถุประสงค์ทางคลินิก เช่น การคัดกรองหรือการคัดแยกผู้ป่วย (screening/triage) เทียบกับการยืนยันการวินิจฉัย

**⭐ แบบฝึกหัดที่ 3 (Exercise 3):** ให้กำหนด `THRESHOLD` เป็นค่าระหว่าง 0 ถึง 1 (ทดลองค่า 0.3 จากนั้นทดลองค่า 0.8 หรือ 0.9) แล้วรันเซลล์ใหม่ และสังเกตการเปลี่ยนแปลงของ confusion matrix
*คำใบ้:* ให้ระบุตัวเลข เช่น `THRESHOLD = 0.3`

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: เปลี่ยนเกณฑ์ตัดสิน (threshold)
THRESHOLD = ...   # TODO: ตัวเลขระหว่าง 0 ถึง 1 (ค่าเริ่มต้นคือ 0.5)

y_pred_t = (test_probs >= THRESHOLD).astype(int)
cm_t = confusion_matrix(y_true, y_pred_t, labels=[0, 1])
ConfusionMatrixDisplay(cm_t, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
plt.title(f"Threshold = {THRESHOLD}"); plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()
tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
print(f"เกณฑ์ {THRESHOLD}: ตรวจไม่พบ (FN)={fn_t}, ผลบวกลวง (FP)={fp_t}, "
      f"ความไว={tp_t/(tp_t+fn_t):.3f}, ความจำเพาะ={tn_t/(tn_t+fp_t):.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
THRESHOLD = 0.3   # เกณฑ์ที่ต่ำลง ซึ่งอาจใช้ในการคัดกรอง (เพื่อตรวจพบผู้ป่วยได้มากขึ้น)

y_pred_t = (test_probs >= THRESHOLD).astype(int)
cm_t = confusion_matrix(y_true, y_pred_t, labels=[0, 1])
ConfusionMatrixDisplay(cm_t, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
plt.title(f"Threshold = {THRESHOLD}"); plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()
tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
print(f"เกณฑ์ {THRESHOLD}: ตรวจไม่พบ (FN)={fn_t}, ผลบวกลวง (FP)={fp_t}, "
      f"ความไว={tp_t/(tp_t+fn_t):.3f}, ความจำเพาะ={tn_t/(tn_t+fp_t):.3f}")

In [ ]:
# ผลลัพธ์ของทุกเกณฑ์ + ROC curve (โค้ดเตรียมไว้แล้ว)
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score
rows = []
for t in [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    p = (test_probs >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, p, labels=[0, 1]).ravel()
    rows.append({"เกณฑ์ (threshold)": t, "ตรวจไม่พบ (FN)": fn_, "ผลบวกลวง (FP)": fp_,
                 "ความไว (sensitivity)": round(tp_/(tp_+fn_), 3), "ความจำเพาะ (specificity)": round(tn_/(tn_+fp_), 3)})
display(pd.DataFrame(rows))

fpr, tpr, _ = roc_curve(y_true, test_probs)
auc = roc_auc_score(y_true, test_probs)
plt.figure(figsize=(4, 4))
plt.plot(fpr, tpr, label=f"ResNet-18 (AUC = {auc:.3f})"); plt.plot([0, 1], [0, 1], "k--", label="coin flip (AUC = 0.5)")
plt.xlabel("1 - specificity (false alarm rate)"); plt.ylabel("Sensitivity"); plt.legend(); plt.title("ROC curve"); plt.show()

💡 **คำอธิบายผลลัพธ์** ตารางแสดงความสัมพันธ์เชิงแลกเปลี่ยน (trade-off) ระหว่างความไวและความจำเพาะ โดยทั่วไปการเพิ่มค่าหนึ่งจะทำให้อีกค่าหนึ่งลดลง **ROC curve** แสดงความสัมพันธ์นี้สำหรับ *ทุก* เกณฑ์ตัดสิน และ **พื้นที่ใต้เส้นโค้ง (ROC AUC)** สรุปความสามารถของโมเดลในการแยกสองกลุ่มโดยไม่ขึ้นกับเกณฑ์ตัดสิน (1.0 = แยกได้สมบูรณ์, 0.5 = เทียบเท่าการสุ่ม) ROC AUC เป็นตัวชี้วัดที่ใช้บนตารางอันดับ 🏆 ด้วย

✍️ **คำถามที่ 3** หากนำโมเดลนี้ไปใช้ **คัดแยก (triage)** ภาพถ่ายรังสีทรวงอกของผู้ป่วยเด็ก (ระบุภาพที่ควรให้รังสีแพทย์อ่านก่อน) ผู้เรียนจะเลือกเกณฑ์ตัดสินใดจากตาราง เพราะเหตุใด และคำตอบจะเปลี่ยนไปหรือไม่ หากใช้โมเดลเพื่อ **ตัดภาวะปอดอักเสบออก (rule out)** โดยไม่มีแพทย์อ่านภาพ จงอธิบาย

✍️ **คำตอบ:**



## ส่วน B — การกำกับข้อมูลภาพและการแบ่งส่วนภาพ (แนวคิดและตัวอย่างโดยสังเขป)

การจำแนกภาพ (classification) ต้องการป้ายกำกับ *ระดับภาพ* ("ภาพนี้มีภาวะปอดอักเสบ") ขณะที่ **การตรวจหาวัตถุ (object detection)** และ **การแบ่งส่วนภาพ (segmentation)** ต้องการป้ายกำกับ *ระดับบริเวณ* ได้แก่ กรอบสี่เหลี่ยม หรือขอบเขตที่แม่นยำ (**mask**) รอบรอยโรค ซึ่งสร้างด้วยเครื่องมือกำกับข้อมูล (annotation tool) เช่น
- **[LabelMe](https://github.com/wkentaro/labelme)** — วาดรูปหลายเหลี่ยมหรือ mask ได้ มีขนาดเล็ก และใช้งานแบบออฟไลน์บนเครื่องของผู้ใช้
- **[CVAT](https://www.cvat.ai/)** — ใช้งานผ่านเว็บ รองรับผู้กำกับข้อมูลหลายคน ขั้นตอนการตรวจทาน และโครงการขนาดใหญ่

การวาด mask เป็นงานที่ใช้เวลามากและต้องอาศัยผู้เชี่ยวชาญ (มักเป็นรังสีแพทย์) ชุดข้อมูลสำหรับ segmentation จึงมักมีขนาดเล็กและมีต้นทุนสูง

เซลล์ด้านล่างแสดงตัวอย่าง segmentation แบบดั้งเดิม (Otsu threshold ซึ่งเลือกจุดตัดความสว่างโดยอัตโนมัติ) เพื่อแสดงลักษณะของ mask โดยไม่ต้องฝึกโมเดล การตรวจหาวัตถุด้วยโมเดลการเรียนรู้ของเครื่อง (machine learning) จะกล่าวถึงใน Lab 5 ผู้เรียนสามารถเปลี่ยนค่า `IMAGE_INDEX` (0–623) เพื่อพิจารณาภาพอื่น

In [ ]:
# การแบ่งส่วนภาพด้วย Otsu threshold (โค้ดเตรียมไว้แล้ว)
from skimage.filters import threshold_otsu
IMAGE_INDEX = 0

img = test_ds[IMAGE_INDEX][0].squeeze().numpy()
thr = threshold_otsu(img)
mask = img > thr          # True = สว่างกว่าจุดตัด

fig, ax = plt.subplots(1, 3, figsize=(10, 3))
ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"X-ray #{IMAGE_INDEX} ({'pneumonia' if y_true[IMAGE_INDEX] else 'normal'})")
ax[1].imshow(mask, cmap="gray"); ax[1].set_title(f"Otsu mask (thr={thr:.2f})")
ax[2].imshow(img, cmap="gray"); ax[2].imshow(mask, alpha=0.3, cmap="Reds"); ax[2].set_title("Overlay")
for a in ax: a.axis("off")
plt.show()

💡 **คำอธิบายผลลัพธ์** mask ระบุทุกพิกเซลที่สว่างกว่าจุดตัดซึ่งเลือกโดยอัตโนมัติ ในภาพถ่ายรังสีทรวงอก วิธีนี้จึงเลือกโครงสร้างที่มีความทึบรังสีเป็นหลัก (กระดูกสันหลัง หัวใจ เนื้อเยื่ออ่อน และกระดูก) ส่วนเนื้อปอดที่มีอากาศปรากฏเป็นสีดำจึง *ไม่* ถูกเลือก กฎอย่างง่ายเช่นนี้ไม่ได้อาศัยความรู้ทางกายวิภาค การแบ่งส่วนปอดหรือรอยโรคที่ถูกต้องจำเป็นต้องใช้โมเดลที่ฝึกด้วย mask ซึ่งผู้เชี่ยวชาญเป็นผู้วาด

✍️ **คำถามที่ 4** หากโรงพยาบาลต้องการระบบปัญญาประดิษฐ์ (AI) ที่ระบุขอบเขตบริเวณ consolidation บนภาพถ่ายรังสีทรวงอก โปรดระบุว่าต้องใช้ป้ายกำกับประเภทใด ผู้ใดควรเป็นผู้กำกับข้อมูล และจงอธิบายปัญหาที่อาจเกิดขึ้นหากผู้กำกับข้อมูลแต่ละคนให้ความเห็นไม่ตรงกัน

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

**ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนของชิ้นงานที่ต้องส่ง** จัดไว้สำหรับผู้เรียนที่ต้องการพัฒนาโมเดลให้มีประสิทธิภาพสูงขึ้น

ให้ส่งค่า P(pneumonia) ของ **ภาพทดสอบทั้ง 624 ภาพ** (`id` = ลำดับที่ 0..623 ในชุด test ของ PneumoniaMNIST) **ตัวชี้วัด: ROC AUC** (ไม่ขึ้นกับเกณฑ์ตัดสิน จึงเป็นข้อมูลที่เสริมความไวและความจำเพาะที่คำนวณไว้แล้ว) หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่าด้านบน แล้วรันเซลล์ดังกล่าวใหม่

> **หลักความซื่อสัตย์ทางวิชาการ (Honour code)** ป้ายกำกับที่ซ่อนไว้มาจากชุดข้อมูลสาธารณะ การค้นหาป้ายกำกับดังกล่าวจึงทำได้ง่ายแต่ไม่มีประโยชน์ต่อการเรียนรู้ ตารางอันดับมีไว้เพื่อสนับสนุนการเรียนรู้ และโน้ตบุ๊กของผู้เรียนจะได้รับการประเมินด้วย ห้ามแก้ไขผลการทำนายด้วยมือ

**Baseline** — เซลล์ด้านล่างส่งผลการทำนายของโมเดลจากส่วนหลักโดยไม่มีการปรับแก้ และสามารถใช้งานได้ทันที

In [ ]:
# 🏆 ส่งผล baseline: ความน่าจะเป็นจากโมเดลในส่วนหลักสำหรับภาพทดสอบ 624 ภาพ
import pandas as pd

def predict_probs(model, loader):
    model.eval(); probs = []
    with torch.no_grad():
        for x, _ in loader:                        # shuffle=False -> ลำดับ = ดัชนีภาพทดสอบ 0..623
            probs += torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().tolist()
    return probs

probs = predict_probs(net, test_loader)
assert len(probs) == 624, "ต้องมีผลการทำนายครบ 624 ภาพ"
pd.DataFrame({"id": range(len(probs)), "prob": probs}).to_csv("lab04_submission.csv", index=False)
print("บันทึกไฟล์ lab04_submission.csv เรียบร้อยแล้ว")
if NAME and JOIN_CODE:
    submit_to_leaderboard("lab04_submission.csv")
else:
    print("ยังไม่ได้ระบุ NAME / JOIN_CODE ในเซลล์การตั้งค่า จึงไม่ได้ส่งผลไปยังตารางอันดับ (ไม่บังคับ)")

### แนวทางการพัฒนาให้ดีกว่า baseline
- เพิ่มจำนวน epoch (เช่น 10–20) และลดอัตราการเรียนรู้ (learning rate) เช่น `lr=3e-4`
- การเพิ่มข้อมูล (data augmentation) ในชุดฝึก เช่น `transforms.RandomRotation(10)`, `transforms.RandomAffine(0, translate=(0.05, 0.05))`
- ใช้ภาพที่มีขนาดใหญ่ขึ้น: `size=64` หรือ `size=128` (MedMNIST+) ซึ่งมีรายละเอียดมากขึ้นแต่ใช้เวลาประมวลผลนานขึ้น
- จัดการความไม่สมดุลของคลาส: `nn.CrossEntropyLoss(weight=...)` หรือ weighted sampler
- เก็บ checkpoint ที่ให้ AUC บนชุด **validation** สูงที่สุด (ห้ามปรับแต่งโมเดลโดยอาศัยชุดทดสอบ)
- ใช้โครงข่ายขนาดใหญ่ขึ้น (ResNet-34/50) หรือ test-time augmentation (เฉลี่ยผลการทำนายจากภาพที่พลิกหรือเลื่อนตำแหน่ง)
- สำหรับการศึกษาเพิ่มเติม: เปลี่ยนเป็น `BreastMNIST` (ภาพอัลตราซาวด์) หรือ `DermaMNIST` (รอยโรคผิวหนัง) โดยแก้ไขเพียงบรรทัดเดียว แล้วเปรียบเทียบระดับความยาก (ไม่ใช้กับตารางอันดับ)

เซลล์ (ไม่บังคับ) ด้านล่างเป็นจุดเริ่มต้น (ภาพ 64×64 + data augmentation + จำนวน epoch มากขึ้น ใช้เวลาประมาณ 5 นาทีบน T4 GPU และนานกว่ามากบน CPU) ผู้เรียนสามารถแก้ไขได้ตามต้องการ

In [ ]:
# 🏆 (ไม่บังคับ) โมเดลที่ปรับปรุงแล้ว — สามารถแก้ไขได้ตามต้องการ แล้วรันใหม่
from sklearn.metrics import roc_auc_score
SIZE, EPOCHS_PLUS, LR = 64, 8, 3e-4

aug_tf = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    transforms.ToTensor(), transforms.Normalize([.5], [.5]),
])
train_plus = DataLoader(PneumoniaMNIST(split="train", transform=aug_tf, download=True, size=SIZE), batch_size=64, shuffle=True)
val_plus   = DataLoader(PneumoniaMNIST(split="val",   transform=tf, download=True, size=SIZE), batch_size=128)
test_plus  = DataLoader(PneumoniaMNIST(split="test",  transform=tf, download=True, size=SIZE), batch_size=128)

net2 = models.resnet18(weights="IMAGENET1K_V1")
w = net2.conv1.weight.data.sum(dim=1, keepdim=True)
net2.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False); net2.conv1.weight.data = w
net2.fc = nn.Linear(net2.fc.in_features, 2)
net2 = train_model(net2.to(device), train_plus, val_plus, EPOCHS_PLUS, lr=LR)

val_y = val_plus.dataset.labels.ravel()
print(f"AUC บนชุด validation: baseline={roc_auc_score(val_y, predict_probs(net, val_loader)):.3f}  "
      f"โมเดลที่ปรับปรุง={roc_auc_score(val_y, predict_probs(net2, val_plus)):.3f}")

probs2 = predict_probs(net2, test_plus)
assert len(probs2) == 624, "ต้องมีผลการทำนายครบ 624 ภาพ"
pd.DataFrame({"id": range(len(probs2)), "prob": probs2}).to_csv("lab04_submission.csv", index=False)
print("บันทึกไฟล์ lab04_submission.csv เรียบร้อยแล้ว (โมเดลที่ปรับปรุง)")
if NAME and JOIN_CODE:
    submit_to_leaderboard("lab04_submission.csv")

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

1. โน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดง confusion matrix ความไวและความจำเพาะ ภาพที่โมเดลจำแนกผิด และตารางเกณฑ์ตัดสิน)
2. คำตอบของ ✍️ **คำถามที่ 1–4** ในเซลล์คำตอบ
3. *(ไม่บังคับ)* ผู้เรียนที่เข้าร่วมโจทย์ท้าทาย 🏆 สามารถระบุชื่อบนตารางอันดับและคะแนนสูงสุดไว้ด้านล่าง

ให้ส่งงานตามช่องทางที่รายวิชากำหนด หากมีข้อสงสัย สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

✍️ **คำตอบ:** *(ไม่บังคับ: ชื่อบนตารางอันดับ / คะแนน)*

